# Task 4: Movie Rating Prediction (Collaborative Filtering)
## 1. Introduction and Data Merging


The goal of this task is to build a model that predicts how a user might rate a movie they have not seen yet. We will use a movie ratings dataset, combining the user ratings file with the movie titles file for better readability.

In [1]:
import pandas as pd

# Loading datasets
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# Merging them
df = pd.merge(ratings, movies, on='movieId')

print(df.head())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


## 2. Data Preprocessing & Sparsity


We must preprocess the data to prepare it for modeling. Recommendation systems often deal with highly sparse matrices (most users haven't rated most movies). We will construct a user-item matrix where rows are users, columns are movies, and the values are the ratings.

In [2]:
# Creation of the User-Item matrix
# Rows = Users, Columns = Movie Titles, Values = Ratings
user_movie_matrix = df.pivot_table(index='userId', columns='title', values='rating')

# matrix check
print(user_movie_matrix.head())

title   '71 (2014)  'Hellboy': The Seeds of Creation (2004)  \
userId                                                        
1              NaN                                      NaN   
2              NaN                                      NaN   
3              NaN                                      NaN   
4              NaN                                      NaN   
5              NaN                                      NaN   

title   'Round Midnight (1986)  'Salem's Lot (2004)  \
userId                                                
1                          NaN                  NaN   
2                          NaN                  NaN   
3                          NaN                  NaN   
4                          NaN                  NaN   
5                          NaN                  NaN   

title   'Til There Was You (1997)  'Tis the Season for Love (2015)  \
userId                                                               
1                             Na

## 3. Building the Collaborative Filtering Model


We will use item-based collaborative filtering. The core idea is that if Movie A and Movie B have similar rating patterns across many users, a user who liked Movie A will likely rate Movie B highly. We will calculate the correlation between movies to find these similarities.

In [3]:
# Picking a popular movie to test our recommendation system.
test_movie = 'eXistenZ (1999)'

# Grab of all user ratings for The Matrix
matrix_user_ratings = user_movie_matrix[test_movie]

# Calculation of correlation of every other movie's ratings to The Matrix's ratings
similar_to_matrix = user_movie_matrix.corrwith(matrix_user_ratings)

# Results in a clean DataFrame and drop movies with no correlation data
corr_matrix = pd.DataFrame(similar_to_matrix, columns=['Correlation'])
corr_matrix.dropna(inplace=True)

print("Raw correlations (Notice how some obscure movies have a perfect 1.0 correlation!):\n", corr_matrix.sort_values('Correlation', ascending=False).head(10))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Raw correlations (Notice how some obscure movies have a perfect 1.0 correlation!):
                                     Correlation
title                                          
Descendants, The (2011)                     1.0
Devil in a Blue Dress (1995)                1.0
Doc Hollywood (1991)                        1.0
Don't Say a Word (2001)                     1.0
Down with Love (2003)                       1.0
Draughtsman's Contract, The (1982)          1.0
Dream Team, The (1989)                      1.0
Crucible, The (1996)                        1.0
Dredd (2012)                                1.0
Drop Dead Gorgeous (1999)                   1.0


## 4. Evaluation and Final Recommendations


Finally, we evaluate the performance by looking at the top correlated results. We must filter out movies that have very few total ratings, as a movie rated 5 stars by only one person might show a false 100% correlation.

In [4]:
# Count of how many total ratings each movie has.
ratings_mean_count = pd.DataFrame(df.groupby('title')['rating'].mean())
ratings_mean_count['rating_counts'] = pd.DataFrame(df.groupby('title')['rating'].count())

# Joining rating counts to our correlation table
corr_matrix = corr_matrix.join(ratings_mean_count['rating_counts'])

# Filtering out movies that have less than 50 total ratings
# And sort by highest correlation
final_recommendations = corr_matrix[corr_matrix['rating_counts'] > 50].sort_values('Correlation', ascending=False)

print(f"\nTop Recommendations if you liked '{test_movie}':")

print(final_recommendations.head(6)[1:])


Top Recommendations if you liked 'eXistenZ (1999)':
                                Correlation  rating_counts
title                                                     
Toy Story 3 (2010)                 1.000000             55
Ratatouille (2007)                 1.000000             72
Guardians of the Galaxy (2014)     1.000000             59
Zombieland (2009)                  0.996116             53
Eraser (1996)                      0.968496             64
